# 05 · Fig. 2 and Section V-C: is the signal final-answer leakage?

Two tests.

* **Span ablation (Fig. 2).** Verifiers are trained on hidden states pooled over
  three views of each candidate: the complete solution, the reasoning with
  everything from the final-answer statement onward removed, and the answer span
  alone. Draw: `Qwen/Qwen2.5-1.5B-Instruct`, chat prompt, T = 0.7, top-p 0.8,
  512 new tokens, K = 8, sampled with vLLM
  (`LVPP_span_recapture_colab.ipynb` regenerates it, about 1.5 h on one GPU).
  The split uses `lvpp.prompts.CHAT_TRIGGER`.
* **Unfinished candidates.** The prefix verifier is evaluated only on candidates
  that are still generating at checkpoint B, so their features cannot contain a
  completed answer statement.

An earlier version of this draw used a trigger regex that never matched
`\boxed{` (`CHAT_TRIGGER_LEGACY`), so for 90.2% of candidates the "reasoning"
view equalled the full solution. The paper reports the corrected draw.

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gpu,gbt] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
#@title Step 1 - trigger rate (seconds)
from lvpp.caches import load_dict_B
te_full = load_dict_B(f'{PATHS["span_root"]}/testfeat.pt', variants=("full",))
te_reas = load_dict_B(f'{PATHS["span_root"]}/testfeat.pt', variants=("reasoning",))
trig = te_full.extra["has_trigger"]
same = float((te_reas.P == te_full.P).all(1).float().mean())
print(f"test candidates with a final-answer statement: {100*np.mean(trig):.1f}%")
print(f"test candidates whose reasoning view equals the full solution: {100*same:.1f}%")

In [ ]:
#@title Step 2 - span ablation, three seeds (about 5 min)
from lvpp.selection import selection_hits
from lvpp.verifier import fit_verifier
from lvpp.stats import paired, clustered_bootstrap_auc_diff
SEEDS = [0, 1, 2]; rows = {}; S = {}; H = {}
for span in ["full", "reasoning", "answer_span"]:
    tr = load_dict_B(f'{PATHS["span_root"]}/pool.pt', variants=(span,))
    te = load_dict_B(f'{PATHS["span_root"]}/testfeat.pt', variants=(span,))
    ok = torch.isfinite(tr.P).all(1) & torch.isfinite(tr.Z).all(1)
    accs, aucs, ss, hs = [], [], [], []
    for s in SEEDS:
        sc = fit_verifier(tr.P[ok], tr.Z[ok], tr.y[ok], seed=s).score(torch.nan_to_num(te.P), torch.nan_to_num(te.Z))
        h = selection_hits(te, "weighted", scores=sc)
        accs.append(100 * h.mean()); aucs.append(roc_auc_score(te.y.numpy(), sc)); ss.append(sc); hs.append(h)
    S[span] = np.mean(ss, 0); H[span] = hs
    rows[span] = float(np.mean(accs)); rows[span + " sd"] = float(np.std(accs)); rows[span + " AUC"] = float(np.mean(aucs))
    print(f"{span:12s} {np.mean(accs):.2f} +/- {np.std(accs):.2f} | AUC {np.mean(aucs):.4f}")
rows["majority"] = 100 * selection_hits(te_full, "majority").mean()
for a in ["reasoning", "answer_span"]:
    ps = [paired(H[a][i], H["full"][i])["p"] for i in range(len(SEEDS))]
    lo, hi = clustered_bootstrap_auc_diff(te_full.y.numpy(), S[a], S["full"])
    print(f"{a} - full: accuracy {rows[a] - rows['full']:+.2f} (McNemar p per seed {np.round(ps, 3)}) | "
          f"AUC {rows[a + ' AUC'] - rows['full AUC']:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}]")
m = trig.astype(bool)
if m.any() and len(set(te_full.y.numpy()[m])) > 1:
    print(f"split candidates only (n={m.sum()}): AUC full {roc_auc_score(te_full.y.numpy()[m], S['full'][m]):.4f}"
          f" | reasoning {roc_auc_score(te_full.y.numpy()[m], S['reasoning'][m]):.4f}")
EXPECTED = {"majority": 76.88, "full": 80.49, "reasoning": 80.36, "answer_span": 79.73,
            "full AUC": 0.8781, "reasoning AUC": 0.8750, "answer_span AUC": 0.8702}
fig2_report = report(rows, EXPECTED, list(EXPECTED), tol=0.3)

In [ ]:
#@title Step 3 - prefix verifier on unfinished candidates only (about 3 min)
from lvpp.caches import load_prefix_C
for B in [110, 169]:
    te = load_prefix_C(PATHS["prefix_root"], "test", B); tr = load_prefix_C(PATHS["prefix_root"], "pool", B)
    fin = np.asarray(te.extra["finished"]).astype(bool); y = te.y.numpy()
    a_all, a_unf = [], []
    for s in [0, 1, 2]:
        sc = fit_verifier(tr.P, tr.Z, tr.y, seed=s).score(te.P, te.Z)
        a_all.append(roc_auc_score(y, sc))
        a_unf.append(roc_auc_score(y[~fin], sc[~fin]) if (~fin).sum() and len(set(y[~fin])) > 1 else np.nan)
    rows[f"unfinished AUC B={B}"] = float(np.nanmean(a_unf))
    print(f"B={B}: finished {100*fin.mean():.1f}% | AUC all {np.mean(a_all):.3f} | "
          f"AUC unfinished only {np.nanmean(a_unf):.3f} (n={int((~fin).sum())})")
EXPECTED.update({"unfinished AUC B=110": 0.812, "unfinished AUC B=169": 0.786})
report(rows, EXPECTED, list(EXPECTED), tol=0.3).to_csv(f"{RESULTS}/fig2.csv", index=False)

In [ ]:
#@title Step 4 - the figure
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "pdf.fonttype": 42, "font.size": 8})
labels = ["complete\nsolution", "reasoning only\n(answer removed)", "answer span\nonly"]
keys = ["full", "reasoning", "answer_span"]; cols = ["#2b6cb0", "#2f855a", "#c05621"]
fig, ax = plt.subplots(1, 2, figsize=(7, 2.2))
for i, k in enumerate(keys):
    ax[0].errorbar(i, rows[k], yerr=rows[k + " sd"], fmt="o", color=cols[i], capsize=3, ms=5)
    ax[0].annotate(f"{rows[k]:.2f}", (i, rows[k]), xytext=(8, -3), textcoords="offset points", fontsize=7)
    ax[1].plot(i, rows[k + " AUC"], "o", color=cols[i], ms=5)
    ax[1].annotate(f"{rows[k + ' AUC']:.3f}", (i, rows[k + " AUC"]), xytext=(8, -3), textcoords="offset points", fontsize=7)
ax[0].axhline(rows["majority"], ls="--", c="k", lw=0.8)
ax[0].text(2.45, rows["majority"] + 0.15, f"majority {rows['majority']:.2f}", ha="right", fontsize=7)
ax[0].set_ylim(np.floor(min(rows["majority"], *(rows[k] for k in keys))) - 1, np.ceil(max(rows[k] for k in keys)) + 1)
ax[0].set_ylabel("selection accuracy (%)")
lo_auc = min(rows[k + " AUC"] for k in keys); hi_auc = max(rows[k + " AUC"] for k in keys)
ax[1].set_ylim(np.floor(lo_auc * 100 - 2) / 100, np.ceil(hi_auc * 100 + 1) / 100); ax[1].set_ylabel("candidate AUC")
for a, t in zip(ax, ["(a)", "(b)"]):
    a.set_xticks(range(3)); a.set_xticklabels(labels); a.set_xlim(-0.5, 2.6); a.set_title(t, fontsize=8)
    a.grid(axis="y", lw=0.3, alpha=0.5)
fig.tight_layout(); fig.savefig(f"{RESULTS}/fig_span_ablation.pdf"); plt.show()